# 02. NVIDIA API payload 구성 실습

목표: DebuggerCafe 글의 `api.py` 구조를 바탕으로 NVIDIA OpenAI-compatible API에 보낼 multimodal message payload를 만든다.

실행 방법:
1. 이 노트북을 위에서 아래로 실행한다.
2. 실제 API 호출은 하지 않는다. API 키 없이도 payload 구조를 검증할 수 있다.
3. 실제 앱을 만들 때는 `pip install -r requirements.txt` 후 `NVIDIA_API_KEY`를 설정한다.

학습 포인트: multimodal API에서 어려운 부분은 모델 호출 한 줄보다 파일 검증, MIME type, data URI, option 구성을 안정적으로 처리하는 일이다.

In [ ]:
import base64
import json
import os
from pathlib import Path
from tempfile import TemporaryDirectory

NVIDIA_API_BASE_URL = "https://integrate.api.nvidia.com/v1"
MODEL = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning"

MIME_TYPES = {
    ".png": "image/png",
    ".jpg": "image/jpeg",
    ".jpeg": "image/jpeg",
    ".mp4": "video/mp4",
    ".mp3": "audio/mpeg",
    ".wav": "audio/wav",
}

CONTENT_TYPES = {
    "image/png": "image_url",
    "image/jpeg": "image_url",
    "video/mp4": "video_url",
    "audio/mpeg": "audio_url",
    "audio/wav": "audio_url",
    "audio/x-wav": "audio_url",
}

## 1. MIME type에서 API content type 찾기

NVIDIA API payload는 파일 종류에 따라 `image_url`, `video_url`, `audio_url` 같은 content type을 써야 한다. 확장자만 믿는 것은 production에서 부족하지만, 입문용 앱에서는 첫 단계로 쓸 수 있다.

In [ ]:
def media_type_from_path_or_mime(path=None, mime=None):
    """파일 경로나 명시 MIME type에서 API content type을 찾는다."""
    ext = Path(path).suffix.lower() if path else ""
    resolved_mime = mime or MIME_TYPES.get(ext)
    return CONTENT_TYPES.get(resolved_mime)


for name in ["diagram.png", "clip.mp4", "voice.wav", "notes.pdf"]:
    print(name, "->", media_type_from_path_or_mime(name))

## 2. local file을 data URI로 변환

DebuggerCafe 예제는 local file을 base64 data URI로 바꿔 API payload에 넣는다. data URI는 실험에는 편하지만 payload가 커지고 로그에 민감정보가 남을 수 있다.

In [ ]:
def file_to_data_uri(filepath):
    """지원되는 local file을 data URI 문자열로 바꾼다."""
    filepath = Path(filepath)
    mime = MIME_TYPES.get(filepath.suffix.lower())
    if mime is None:
        raise ValueError(f"Unsupported file type: {filepath.suffix.lower()}")
    encoded = base64.b64encode(filepath.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{encoded}"


with TemporaryDirectory() as temp_dir:
    sample_path = Path(temp_dir) / "tiny.png"
    # PNG header 일부만 가진 작은 더미 파일이다. API 전송용 실제 이미지로는 충분하지 않지만 data URI 구조 학습에는 충분하다.
    sample_path.write_bytes(b"\x89PNG\r\n\x1a\n")
    uri = file_to_data_uri(sample_path)
    print(uri[:40] + "...")

## 3. file part 만들기

실제 Gradio 입력은 문자열 경로, dict, `.path` 속성을 가진 객체 등 여러 형태로 들어올 수 있다. 그래서 입력을 정규화하는 함수가 필요하다.

In [ ]:
def get_file_path(file):
    if isinstance(file, (str, Path)):
        return file
    if isinstance(file, dict):
        return file.get("path") or file.get("name") or file.get("orig_name")
    return getattr(file, "path", None)


def build_file_part(file):
    """NVIDIA/OpenAI-compatible content part를 만든다."""
    url = None
    mime = None
    if isinstance(file, dict):
        url = file.get("url")
        mime = file.get("mime_type") or file.get("mime") or file.get("type")

    filepath = get_file_path(file)
    media_type = media_type_from_path_or_mime(filepath, mime)

    if url and str(url).startswith(("data:", "http://", "https://")):
        if media_type is None:
            return None
        return {"type": media_type, media_type: {"url": url}}

    if filepath is None:
        return None
    if media_type is None:
        raise ValueError(f"Unsupported file type: {Path(filepath).suffix.lower()}")
    return {"type": media_type, media_type: {"url": file_to_data_uri(filepath)}}

In [ ]:
remote_file = {"url": "https://example.com/demo.mp4", "mime_type": "video/mp4"}
build_file_part(remote_file)

## 4. user message 만들기

OpenAI-compatible multimodal payload에서 한 user message의 `content`는 여러 part의 배열이다. media part를 먼저 넣고, 마지막에 text part를 넣는 흐름을 사용한다.

In [ ]:
def build_user_message(text, files=None):
    content = []
    for file in files or []:
        file_part = build_file_part(file)
        if file_part is not None:
            content.append(file_part)

    if text and text.strip():
        content.append({"type": "text", "text": text})

    if not content:
        raise ValueError("At least one text or media input is required")

    return {"role": "user", "content": content}


message = build_user_message("Summarize this video and mention any spoken instructions.", [remote_file])
print(json.dumps(message, indent=2)[:500])

## 5. reasoning mode와 video audio option

원문 예제는 `extra_body`로 `enable_thinking`, `reasoning_budget`, `use_audio_in_video`를 전달한다. 이 값은 답변 품질뿐 아니라 latency와 비용에도 영향을 준다.

In [ ]:
def build_extra_body(enable_thinking=False, use_audio_in_video=False, reasoning_budget=16384):
    extra_body = {"chat_template_kwargs": {"enable_thinking": bool(enable_thinking)}}
    if enable_thinking:
        extra_body["reasoning_budget"] = reasoning_budget
    if use_audio_in_video:
        extra_body["mm_processor_kwargs"] = {"use_audio_in_video": True}
    return extra_body


build_extra_body(enable_thinking=True, use_audio_in_video=True)

## 6. 최종 request preview

실제 OpenAI SDK 호출은 `client.chat.completions.create(...)`가 맡는다. 여기서는 전송 직전의 구조만 만든다.

In [ ]:
def build_request_preview(messages, enable_thinking, use_audio_in_video):
    return {
        "base_url": NVIDIA_API_BASE_URL,
        "model": MODEL,
        "messages": messages,
        "temperature": 0.6 if enable_thinking else 0.2,
        "top_p": 0.95 if enable_thinking else None,
        "max_tokens": 4096,
        "stream": True,
        "extra_body": build_extra_body(enable_thinking, use_audio_in_video),
        "has_api_key": bool(os.getenv("NVIDIA_API_KEY")),
    }


request = build_request_preview([message], enable_thinking=True, use_audio_in_video=True)
print(json.dumps(request, indent=2)[:900])

## 정리

- multimodal request는 text와 media part 배열로 구성된다.
- local file은 data URI로 바꿀 수 있지만 production에서는 크기와 보안 문제가 있다.
- reasoning mode와 video audio option은 `extra_body`로 전달한다.
- 실제 앱에서는 MIME sniffing, 파일 크기 제한, timeout, retry, rate limit 처리가 필요하다.